<div dir="rtl" align="right">

# مُعالجةُ المخزنِ المؤقتِ في الوقتِ الحقيقيِّ - النافذةُ المنزلقةُ

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

يُوضّحُ هذا الدفترُ **مخزنًا مؤقتًا منزلقًا** لمحاكاةِ معالجةِ EEG في الوقتِ الحقيقيِّ. نَمرُّ على إشارةِ القناةِ P4 في حزمٍ من 50 عينةً، مع الحفاظِ على مخزنٍ مؤقتٍ منزلقٍ من 500 عينةٍ (2.5 ثانيةٍ). في كلِّ خطوةٍ، نُطبّقُ مُرشِّحَ متوسطٍ متحرّكٍ (نافذةٌ = 11) على المخزنِ ونُسجّلُ الناتجَ المُرشَّحَ.

## ماذا يَفعلُ هذا الدفترُ

1. يحمّلُ قناةَ P4 من مجموعةِ بياناتِ EEG المحليةِ
2. يُحاكي المعالجةَ في الوقتِ الحقيقيِّ بحزمٍ من 50 عينةٍ
3. يُحافظُ على مخزنٍ مؤقتٍ منزلقٍ من 500 عينةٍ (2.5 ثانيةٍ)
4. يُطبّقُ مُرشِّحَ متوسطٍ متحرّكٍ (نافذةٌ = 11) في كلِّ خطوةٍ
5. يُسجّلُ الناتجَ المُرشَّحَ التدريجيَّ

## المُخرجاتُ المُتوقّعةُ

- المُخطّطُ العلويُّ يُظهرُ **الإشارةَ الخامَ الأصليةَ** (أولُ 5000 عينةٍ) باللونِ الأزرقِ
- المُخطّطُ السفليُّ يُظهرُ **الناتجَ المُرشَّحَ في الوقتِ الحقيقيِّ** باللونِ الأحمرِ، مُوضحًا الترشيحَ التدريجيَّ
- خطٌّ عموديٌّ متقطّعٌ يُحدّدُ الموضعَ "الحاليَّ" عند العينةِ 2500
- الناتجُ المُرشَّحُ أنعمُ من الأصليِّ، مُظهرًا تأثيرَ المتوسطِ المتحرّكِ

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | الوصفُ |
| --------- | ------- | ------ |
| FS | 200 Hz | معدّلُ أخذِ العيناتِ |
| CHUNK_SIZE | 50 | العيناتُ في كلِّ حزمةِ معالجةٍ |
| BUFFER_SIZE | 500 | طولُ المخزنِ المنزلقِ (2.5 ثانيةٍ) |
| MA_WINDOW | 11 | حجمُ نافذةِ المتوسطِ المتحرّكِ |
| N_PLOT | 5000 | عددُ العيناتِ للرسمِ |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install scipy numpy plotly wfdb

<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')

In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1

<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2، قناةَ P4 (المنطقةُ الجداريةُ).

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')

<div dir="rtl" align="right">

## 4. تطبيقُ المعالجةِ

نُحاكي المعالجةَ في الوقتِ الحقيقيِّ بالمرورِ على الإشارةِ في حزمٍ من 50 عينةٍ. يُحافَظُ على مخزنٍ مؤقتٍ منزلقٍ من 500 عينةٍ (2.5 ثانيةٍ)، ويُطبَّقُ مُرشِّحُ المتوسطِ المتحرّكِ (نافذةٌ = 11) في كلِّ خطوةٍ.

</div>

In [ ]:
CHUNK_SIZE = 50
BUFFER_SIZE = 500
MA_WINDOW = 11
N_PLOT = 5000
CURRENT_POS = 2500

n_samples = min(N_PLOT, len(channel_data))
signal_plot = channel_data[:n_samples]

def moving_average(data, window):
    if len(data) < window:
        return data.copy()
    kernel = np.ones(window) / window
    padded = np.pad(data, (window // 2, window - 1 - window // 2), mode='edge')
    return np.convolve(padded, kernel, mode='valid')

buffer = np.zeros(BUFFER_SIZE)
buffer_fill = 0
filtered_output = np.zeros(n_samples)

for start in range(0, n_samples, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, n_samples)
    chunk = signal_plot[start:end]
    for i, sample in enumerate(chunk):
        if buffer_fill < BUFFER_SIZE:
            buffer[buffer_fill] = sample
            buffer_fill += 1
        else:
            buffer = np.roll(buffer, -1)
            buffer[-1] = sample
        filtered_buffer = moving_average(buffer[:buffer_fill], MA_WINDOW)
        filtered_output[start + i] = filtered_buffer[-1]

print(f'Processed {n_samples} samples in chunks of {CHUNK_SIZE}')
print(f'Buffer size: {BUFFER_SIZE} samples ({BUFFER_SIZE/fs:.1f} seconds)')
print(f'Moving average window: {MA_WINDOW}')

<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**
- المُخطّطُ العلويُّ يُظهرُ الإشارةَ الخامَ بكلِّ ضجيجها وتقلّباتِها
- المُخطّطُ السفليُّ يُظهرُ الناتجَ المُرشَّحَ تدريجيًّا — لاحظْ أنه أنعمُ
- الخطُّ العموديُّ المتقطّعُ عند العينةِ 2500 يُمثّلُ موضعَ المعالجةِ "الحاليَّ"
- مُرشِّحُ المتوسطِ المتحرّكِ يُقلّلُ الضجيجَ عاليَ الترددِ مع الحفاظِ على الاتجاهِ العامِّ

</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

x = np.arange(n_samples)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Original Signal (P4)', 'Real-time Filtered Output (Moving Average)'))

fig.add_trace(go.Scatter(x=x, y=signal_plot, name='Original',
                         line=dict(color='blue', width=0.5)), row=1, col=1)
fig.add_vline(x=CURRENT_POS, line_dash='dash', line_color='black',
              annotation_text='Current position', row=1, col=1)

fig.add_trace(go.Scatter(x=x, y=filtered_output, name='Filtered',
                         line=dict(color='red', width=0.5)), row=2, col=1)
fig.add_vline(x=CURRENT_POS, line_dash='dash', line_color='black',
              annotation_text='Current position', row=2, col=1)

fig.update_layout(height=600, title_text='Real-time Buffer Processing - Sliding Window',
                  xaxis2_title='Sample index', yaxis_title='Amplitude (uV)',
                  yaxis2_title='Amplitude (uV)')
fig.show()

<div dir="rtl" align="right">

## خلاصةٌ

- يُتيحُ **المخزنُ المؤقتُ المنزلقُ** المعالجةَ في الوقتِ الحقيقيِّ بالاحتفاظِ بنافذةٍ ثابتةِ الحجمِ من أحدثِ العيناتِ
- المعالجةُ في **حزمٍ** (50 عينةً) تُحاكي كيفيةَ تعاملِ الأنظمةِ الحقيقيةِ مع البياناتِ الواردةِ
- **مُرشِّحُ المتوسطِ المتحرّكِ** يُنعّمُ الإشارةَ بمتوسّطِ العيناتِ المُجاورةِ
- يَمتلئُ المخزنُ تدريجيًّا، لذا للعيناتِ المُبكّرةِ تأثيرُ ترشيحٍ أقلُّ
- هذا النهجُ هو أساسُ أنظمةِ مراقبةِ EEG في الوقتِ الحقيقيِّ

</div>